# Story 02 — KNN Anonymization for Mixed Variables

**Phase 3 Experimentation** | Bank Customer Churn dataset

This folder implements **User Story 02** end-to-end using notebooks:

| Notebook | Purpose |
|----------|---------|
| `00_overview.ipynb` | Dataset profile, k-anonymity analysis, experiment design |
| `01_run_experiments.ipynb` | Run all config iterations → `iterations/<name>/` |
| `02_compare_results.ipynb` | Aggregate scorecards, trade-offs, Gate 3 recommendation |

**Shared library:** `knn_lib.py` (imported by notebooks)

**Story alignment:**
- Step A: k-anonymity (k=5) → identify suppressed records
- Steps B–E: Mixed KNN synthesis replaces suppressed rows only
- Step F: TVD (categorical), KS (numerical), utility, privacy, scorecard


In [1]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

STORY_ROOT = Path(".").resolve()
if str(STORY_ROOT) not in sys.path:
    sys.path.insert(0, str(STORY_ROOT))

from knn_lib import (
    load_dataset, dataset_profile, identify_suppressed,
    DATA_PATH, QI_COLS, CATEGORICAL_COLS, NUMERICAL_COLS,
    EXPERIMENT_GRID, TVD_THRESHOLD, KS_THRESHOLD,
)

sns.set_style("whitegrid")
print("Data:", DATA_PATH)


Data: /home/neosoft/KNN_demo/Bank Customer Churn Prediction.csv


## Area 1 — Dataset Profile


In [2]:
df = load_dataset()
profile = dataset_profile(df)
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
profile


Rows: 10,000 | Columns: 12


,column,dtype,role,missing_pct,n_unique,cardinality_ratio,high_cardinality
0,customer_id,int64,id,0.0,10000,1.0000,True
1,credit_score,int64,numerical,0.0,460,0.0460,False
2,country,str,categorical,0.0,3,0.0003,False
3,gender,str,categorical,0.0,2,0.0002,False
4,age,int64,numerical,0.0,70,0.0070,False
5,tenure,int64,numerical,0.0,11,0.0011,False
6,balance,float64,numerical,0.0,6382,0.6382,True
7,products_number,int64,numerical,0.0,4,0.0004,False
8,credit_card,int64,numerical,0.0,2,0.0002,False
9,active_member,int64,numerical,0.0,2,0.0002,False


In [3]:
# Feature mix
roles = profile["role"].value_counts()
cat_ratio = roles.get("categorical", 0) / len(profile)
num_ratio = roles.get("numerical", 0) / len(profile)
print(f"Categorical features: {len(CATEGORICAL_COLS)} | Numerical: {len(NUMERICAL_COLS)}")
print(f"High-cardinality flags: {profile[profile['high_cardinality']][['column','n_unique','cardinality_ratio']].to_string(index=False) or 'none'}")


Categorical features: 2 | Numerical: 8
High-cardinality flags:           column  n_unique  cardinality_ratio
     customer_id     10000             1.0000
         balance      6382             0.6382
estimated_salary      9999             0.9999


## Step A — K-Anonymity & Suppression (k=5)


In [4]:
suppressed, suppression_summary = identify_suppressed(df, k=5)
suppression_summary


,n_rows,k_anonymity,n_suppressed_tuple,pct_suppressed_tuple,n_single_column_unique,pct_single_column_unique
0,10000,5,5551,55.51,4,0.04


In [5]:
print(f"Suppressed (need synthesis): {suppressed.sum():,} ({100*suppressed.mean():.1f}%)")
print(f"Non-suppressed (neighbour pool): {(~suppressed).sum():,}")


Suppressed (need synthesis): 5,551 (55.5%)
Non-suppressed (neighbour pool): 4,449


## Experiment Grid Preview


In [6]:
grid_df = pd.DataFrame([{
    "folder": c.folder, "name": c.name, "k_neighbors": c.k_neighbors,
    "cat_gen": c.cat_gen_method, "num_gen": c.num_gen_method,
    "scaler": c.scaler_method, "description": c.description,
} for c in EXPERIMENT_GRID])
grid_df


,folder,name,k_neighbors,cat_gen,num_gen,scaler,description
0,01_baseline_kanon5_knn15_mixed_weighted_mode_s...,Baseline,15,weighted_mode,interpolation,standard,"k=5 anon, K=15 neighbours, mixed distance, wei..."
1,02_knn_neighbors_k10_weighted_mode_standard,Low K neighbours,10,weighted_mode,interpolation,standard,Smaller neighbour pool (K=10)
2,03_knn_neighbors_k25_weighted_mode_standard,High K neighbours,25,weighted_mode,interpolation,standard,Larger neighbour pool (K=25)
3,04_cat_generation_probability_sampling,Probability categorical,15,probability,interpolation,standard,Probability sampling for categoricals
4,05_cat_generation_mode,Mode categorical,15,mode,interpolation,standard,Plain mode for categoricals
5,06_num_generation_weighted_mean,Weighted mean numerical,15,weighted_mode,weighted_mean,standard,Distance-weighted mean for numerics
6,07_scaler_robust,Robust scaler,15,weighted_mode,interpolation,robust,RobustScaler for numerical features
7,08_distance_numerical_heavy_2x,Numerical-heavy distance,15,weighted_mode,interpolation,standard,"NUM_WEIGHT=2, CAT_WEIGHT=0.5"
8,09_distance_categorical_heavy_2x,Categorical-heavy distance,15,weighted_mode,interpolation,standard,"NUM_WEIGHT=0.5, CAT_WEIGHT=2"
9,10_seed_stability_seed123,Seed stability,15,weighted_mode,interpolation,standard,"Same config, different seed (robustness check)"


## Next step

Run **`01_run_experiments.ipynb`** to execute all iterations.
Each iteration writes to `iterations/<folder_name>/`:
- `anonymized_dataset.csv`
- `category_metrics.csv` (TVD per column)
- `numeric_metrics.csv` (KS per column)
- `scorecard.csv`
- `summary.json`
